# Airbnb - Data Visualizations Project

## Investor Approach

This jupiter file is also available on:
https://github.com/gsonego/airbnb_analysis/blob/main/Investor/Airbnb_Investor.ipynb


In [ ]:
# First of all, we need to import pandas library
import pandas as pd
import numpy as np
import re # for the text count in amenities

In [ ]:

# Then we can try to read the CSV file for the first time
df = pd.read_csv('../listings.csv')
df.info()
df.head(5)

In [ ]:
# Before we start our analysis, let's remove any unnecessary columns
# Test if columns exist before dropping them
columns_to_drop = ['scrape_id', 'last_scraped', 'source', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'requires_license', 'license', 'region_id', 'region_parent_id', 'region_parent_parent_id', 'region_parent_parent_name']
existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
df = df.drop(columns=existing_columns_to_drop)

df.info()

In [ ]:
# Check the unique values in the 'region_parent_name' column
df['region_parent_name'].unique()

In [ ]:
# Lets do some data manipulation to transform the 'region_parent_name' into counties
df['region_parent_name'] = df['region_parent_name'].str.replace('City And County Council', '').str.strip()
df['region_parent_name'] = df['region_parent_name'].str.replace('County Council', '').str.strip()
df['region_parent_name'] = df['region_parent_name'].str.replace('City Council', '').str.strip()
df['region_parent_name'] = df['region_parent_name'].str.replace('South Dublin', 'Dublin').str.strip()
df['region_parent_name'] = df['region_parent_name'].str.replace('Dun Laoghaire-rathdown', 'Dublin').str.strip()

# Check the unique values in the 'county' column
df['region_parent_name'].unique()

In [ ]:
# Peek at the cleaned dataset
df.head(5)

In [ ]:
# Check for missing values
missing_values = df['price'].isnull().sum()
print(f"Missing values in 'price': {missing_values} ({missing_values/len(df)*100:.2f}%)")

# Step 3: Check data type
print(f"Data type of 'price': {df['price'].dtype}")

In [ ]:
# Let's fill any missing values in the 'price' column with 0
df['price'] = df['price'].fillna(0)

# Let's fill any mission values for bedrooms and bathrooms with 0
df['bedrooms'] = df['bedrooms'].fillna(0)
df['bathrooms'] = df['bathrooms'].fillna(0)

# Fill is super host missing values with 'f'
df['host_is_superhost'] = df['host_is_superhost'].fillna('f')

# Revenue Density (Divide the estimated revenue in last 365 days by the number of people a listing can accommodate)
df['revenue_density'] = df['estimated_revenue_l365d'] / df['accommodates'].replace(0, np.nan)

In [ ]:
# Check for non-numeric values (strings with '$' or commas)
non_numeric = df['price'].apply(lambda x: not isinstance(x, (int, float)) and pd.notnull(x))
if non_numeric.any():
    print("Non-numeric values found in 'price':")
    print(df[non_numeric]['price'].head())
    # Clean price column if needed (remove '$', commas, and convert to float)
    df['price'] = df['price'].replace(r'[\$,]', '', regex=True).astype(float)
    print("Cleaned 'price' column to numeric format.")

In [ ]:
# Verify the data type again
print(f"Data type of 'price': {df['price'].dtype}")

In [ ]:
# Use a function to count amenities items in each string
def count_amenity_items(amenities): 
    # Finds all substrings within double quotes
    return len(re.findall(r'"(.*?)"', str(amenities))) # Len is used to count how many strings were found in each. Make sure that amenities is a string, read strings using regular expression function and use .? to catch anything between double quotes

df['amenities_count'] = df['amenities'].apply(count_amenity_items) # If the above works, create a new column

# Let's drop any rows where 'price' is less than or equal to 0
# But let'set's first check how many such rows exist
invalid_price_count = (df['price'] <= 0).sum()
print(f"Rows with 'price' <= 0: {invalid_price_count} ({invalid_price_count/len(df)*100:.2f}%)")

# Now we can safely drop those rows
df = df[df['price'] > 0]

In [ ]:
# Now let's check for outliers
print(f"Price statistics:\n{df['price'].describe()}")

In [ ]:
# Let's delete any rows where the price is above 1000
outlier_count = (df['price'] > 1000).sum()
print(f"Rows with 'price' > 1000: {outlier_count} ({outlier_count/len(df)*100:.2f}%)")

df = df[df['price'] <= 1000]

In [ ]:
# fill empty beds with 1
df['bedrooms'] = df['bedrooms'].fillna(1)

# replace 0 beds with 1
df['bedrooms'] = df['bedrooms'].replace(0, 1)

# ------------------

# fill empty beds with 1
df['beds'] = df['beds'].fillna(1)

# replace 0 beds with 1
df['beds'] = df['beds'].replace(0, 1)

# ------------------

# fill empty beds with 1
df['bathrooms'] = df['bathrooms'].fillna(1)

# replace 0 beds with 1
df['bathrooms'] = df['bathrooms'].replace(0, 1)

In [ ]:
# Have a look at the final dataframe info
df.info()

In [ ]:
# Save the cleaned (and final) dataset to a new CSV file
df.to_csv('Ireland_Airbnb_Listing__Detailed__Investor.csv', index=False)